In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn

PROJECT_ROOT = Path.cwd().parents[0]
sys.path.insert(0, str(PROJECT_ROOT))

from model.transformer_class import RNATransformer
from model.utils import center_coords_multi, combined_loss_multi
from model.dataset_loader import RNADataset, RNATestDataset, rna_collate_fn, rna_test_collate_fn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import plotly.graph_objects as go

plt.style.use("default")
plt.rcParams["figure.figsize"] = (15,6)
matplotlib.rcParams['axes.labelsize'] = 14
matplotlib.rcParams['xtick.labelsize'] = 12
matplotlib.rcParams['ytick.labelsize'] = 12
matplotlib.rcParams['text.color'] = 'k'

# Loading data

In [ ]:
train_sequence = pd.read_csv(
    "C:/Users/Admin/Downloads/stanford-rna-3d-folding-2/train_sequences.csv"
)
train_labels = pd.read_csv(
    "C:/Users/Admin/Downloads/stanford-rna-3d-folding-2/train_labels.csv"
)
validation_sequence = pd.read_csv(
    "C:/Users/Admin/Downloads/stanford-rna-3d-folding-2/validation_sequences.csv"
)
validation_labels = pd.read_csv(
    "C:/Users/Admin/Downloads/stanford-rna-3d-folding-2/validation_labels.csv"
)

test_sequence = pd.read_csv(
    "C:/Users/Admin/Downloads/stanford-rna-3d-folding-2/test_sequences.csv"
)

In [ ]:
train_sequence["len_sequence"] = train_sequence["sequence"].str.len()

In [ ]:
train_sequence.head()

In [ ]:
train_labels.head()

In [ ]:
print("number of unique target IDs which are pdb_id_chain_id", train_sequence["target_id"].nunique())

In [ ]:
# distribution of sequence lengths:


fig, ax = plt.subplots()

ax.hist(train_sequence["len_sequence"], bins=500, linewidth=0.5, edgecolor="white")
ax.set_yscale('log', base=10)
ax.set_xscale('log', base=10)
plt.xlabel("sequence_length")
plt.ylabel("frequency")
plt.title("sequence length distribution in log scale")
plt.show()


In [ ]:
train_labels[["split_ID", "suffix"]] = train_labels["ID"].str.split('_', expand=True)

In [ ]:
# plotting some target examples
unique_ids = train_labels["split_ID"].unique()
for target_id in unique_ids[:10]:
    # target_id = train_labels["ID"].iloc[i].split("_")[0]
    sub = train_labels[train_labels["ID"].str.startswith(target_id)]

    x = sub["x_1"].values
    y = sub["y_1"].values
    z = sub["z_1"].values


    fig = go.Figure(
        data=[go.Scatter3d(
            x=x, y=y, z=z,
            mode="lines+markers",
            marker=dict(size=3),
            line=dict(width=2)
        )]
    )

    fig.update_layout(
        title=f"RNA structure: {target_id}",
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="Z"
        )
    )

    fig.show()



In [ ]:

lengths = train_sequence["sequence"].str.len()

print("sequence length percentiles:", np.percentile(lengths, [50, 75, 90, 95, 99]))

In [ ]:
# Reducing long sequences examples for initial model run

# filtered_train_sequence = train_sequence[train_sequence["sequence"].str.len() <2500]

valid_targets = set(
    train_sequence.loc[train_sequence.len_sequence < 2000, "target_id"]
)

train_sequences = train_sequence[
    train_sequence.target_id.isin(valid_targets)
]

train_labels = train_labels[
    train_labels.split_ID.isin(valid_targets)
]

In [ ]:
from torch.utils.data import DataLoader

dataset = RNADataset(train_sequences, train_labels)

# Checking correct dataset creation: sequence should be of shape (L,), and each nucleotide should have 3 coordinates, which makes shape (L,3)
seq, coords = dataset[0]

print(seq.shape)     # (L,)
print(coords.shape)  # (L, 3)

In [ ]:
dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=rna_collate_fn
)

# Modeling

# Encoder Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
model = RNATransformer()
model = model.to(device)
num_steps = 2 * len(dataloader)
n_epochs = 5

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=n_epochs,
    eta_min=1e-5
)

batch = next(iter(dataloader))

tokens = batch["tokens"].to(device)
coords = batch["coords"].to(device)
mask = batch["mask"].to(device)

pred, Z = model(tokens)

print(pred.shape)     # (B, L, 3)
print(coords.shape)   # (B, L, 3)

In [ ]:
model.train()
for epoch in range(n_epochs):
    for step, batch in enumerate(dataloader):
        tokens = batch["tokens"].to(device)
        coords = batch["coords"].to(device)
        mask = batch["mask"].to(device)

        pred, Z = model(tokens)

        coords_centered = center_coords_multi(coords, mask)
        pred_centered   = center_coords_multi(pred, mask)

        loss = combined_loss_multi(pred_centered, coords_centered, mask, w_dist=0.2, Z=Z, w_smooth = 0.001, w_aux=0.01)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        

        if step % 20 == 0:
            lr = scheduler.get_last_lr()[0]
            print(f"Step {step} | Loss {loss.item():.4f} | LR {lr:.6f}")   
            continue
    scheduler.step()
    

In [ ]:
from torch.utils.data import DataLoader

test_dataset = RNATestDataset(test_sequence)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,          # IMPORTANT for submission simplicity
    shuffle=False,
    collate_fn=rna_test_collate_fn
)

In [ ]:
model.eval()

all_predictions = []

with torch.no_grad():
    # disables dropout (if any)
    # disables gradient tracking
    # saves memory

    for batch in test_loader:
        tokens = batch["tokens"].to(device)       # (1, L)
        mask = batch["mask"].to(device)          # (1, L)
        meta = batch["meta"][0]        # dict

        pred = model(tokens)           # (1, L, 3)
        pred = pred.squeeze(0)         # (L, 3)

        L = meta["length"]

        all_predictions.append({
            "target_id": meta["target_id"],
            "coords": pred[:L].cpu().numpy()  # (L, 3)
        })

In [ ]:
# Create submission
submission_rows = []

for item in all_predictions:
    target_id = item["target_id"]
    coords = item["coords"]      # (L, 3)

    # Get original sequence from test_sequences_df
    seq = test_sequence.loc[
        test_sequence["target_id"] == target_id, "sequence"
    ].values[0]

    for i, nucleotide in enumerate(seq):
        row = {
            "ID": f"{target_id}_{i+1}",
            "resname": nucleotide,
            "resid": i + 1,
        }

        # repeat same coords for x_1..x_5
        for k in range(1, 6):
            row[f"x_{k}"] = float(coords[i, 0])
            row[f"y_{k}"] = float(coords[i, 1])
            row[f"z_{k}"] = float(coords[i, 2])

        submission_rows.append(row)

submission_df = pd.DataFrame(submission_rows)

submission_df.to_csv("submission.csv", index=False)